# Nexus Bots — Qwen3-0.6B Fine-Tuning (QLoRA)

Fine-tune **Qwen3-0.6B** for domain-specific function calling in robotics e-commerce.

**Requirements:** Colab T4 GPU (free tier), ~4 hours training time.

| Param | Value |
|-------|-------|
| Model | Qwen3-0.6B |
| Method | QLoRA (4-bit NF4) |
| Rank | 16 |
| Alpha | 32 |
| Epochs | 3 |
| LR | 2e-4 |
| Dataset | 900 train / 100 test |

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## 2. Mount Drive & Upload Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nexus-bots'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

# Upload train.jsonl and test.jsonl to DRIVE_DIR or use files.upload()
print('Upload finetune/data/train.jsonl and test.jsonl to:', DRIVE_DIR)

In [ ]:
# If files aren't on Drive yet, upload directly:
from google.colab import files
import shutil

# Uncomment to upload:
# uploaded = files.upload()  # Select train.jsonl and test.jsonl
# for name in uploaded:
#     shutil.move(name, f'{DRIVE_DIR}/{name}')

TRAIN_PATH = f'{DRIVE_DIR}/train.jsonl'
TEST_PATH = f'{DRIVE_DIR}/test.jsonl'

assert os.path.exists(TRAIN_PATH), f'Train file not found at {TRAIN_PATH}'
print(f'Train: {TRAIN_PATH}')
print(f'Test: {TEST_PATH}')

## 3. Configuration

In [ ]:
import json

# ── Model config ──
MODEL_ID = 'unsloth/Qwen3-0.6B'  # Unsloth-optimized mirror
MAX_SEQ_LENGTH = 1024

# ── QLoRA config ──
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

# ── Training config ──
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
WARMUP_RATIO = 0.03
SAVE_STEPS = 200

# ── Output ──
OUTPUT_DIR = f'{DRIVE_DIR}/checkpoints'
ADAPTER_DIR = f'{DRIVE_DIR}/qwen3-0_6b-fc-v1'
MERGED_DIR = f'{DRIVE_DIR}/qwen3-0_6b-fc-v1-merged'

print('Config loaded.')

## 4. Load Model + Apply QLoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.get_model(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'Model loaded: {MODEL_ID}')
print(f'Trainable params: {model.print_trainable_parameters()}')

## 5. Load Dataset

In [ ]:
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, 'r') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_raw = load_jsonl(TRAIN_PATH)
test_raw = load_jsonl(TEST_PATH)

# Extract messages for SFTTrainer
train_dataset = Dataset.from_list([{'messages': r['messages']} for r in train_raw])
eval_dataset = Dataset.from_list([{'messages': r['messages']} for r in test_raw])

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print(f'\nSample system prompt (first 200 chars):')
print(train_raw[0]['messages'][0]['content'][:200])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=SAVE_STEPS,
    max_seq_length=MAX_SEQ_LENGTH,
    seed=42,
    report_to='none',
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
)

print(f'Starting training: {NUM_EPOCHS} epochs, effective batch {BATCH_SIZE * GRAD_ACCUM}')
trainer.train()

## 7. Save Adapter

In [ ]:
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved → {ADAPTER_DIR}')

## 8. Save Merged fp16 (Optional)

In [ ]:
os.makedirs(MERGED_DIR, exist_ok=True)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged model saved → {MERGED_DIR}')

## 9. Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and set your HF token:
# HF_TOKEN = 'hf_...'
# HF_REPO = 'nexus-bots/qwen3-0_6b-fc-v1'
# model.push_to_hub(HF_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
# print(f'Pushed to {HF_REPO}')

## 10. Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

# System prompt (same as training)
system_prompt = train_raw[0]['messages'][0]['content']

test_queries = [
    ('Show me drones under 50000', '{"current_page":"catalog","active_category":"Drone"}'),
    ('mera order kahan hai', '{"current_page":"home"}'),
    ('naku kitchen robots chupinchandi', '{"current_page":"catalog"}'),
]

for query, ctx in test_queries:
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'Query: {query}\nContext: {ctx}'},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\nQ: {query}')
    print(f'A: {generated.strip()}')
    print('-' * 60)

## 11. Training Metrics

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_losses = [(m['step'], m['loss']) for m in logs if 'loss' in m]
eval_losses = [(m['step'], m['eval_loss']) for m in logs if 'eval_loss' in m]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label='Train Loss', alpha=0.8)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label='Eval Loss', alpha=0.8, marker='o')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Qwen3-0.6B Fine-Tuning Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/training_loss.png', dpi=150)
plt.show()

print(f'Final train loss: {train_losses[-1][1]:.4f}' if train_losses else 'No train logs')
print(f'Final eval loss: {eval_losses[-1][1]:.4f}' if eval_losses else 'No eval logs')